In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import re

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- 1. DEFINE VOCABULARY CLASS (Must match original definition) ---
class Vocabulary:
    def __init__(self, max_size=5000):
        self.max_size = max_size
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.word2idx = {self.pad_token: 0, self.unk_token: 1}
        self.idx2word = {0: self.pad_token, 1: self.unk_token}
        
    def tokenize(self, text):
        return re.findall(r'\b\w+\b', text.lower())

    def encode(self, text, max_len=36):
        tokens = self.tokenize(text)
        indices = [self.word2idx.get(w, self.word2idx[self.unk_token]) for w in tokens]
        if len(indices) < max_len:
            indices = indices + [self.word2idx[self.pad_token]] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        return indices

# --- 2. DEFINE BILSTM MODEL ARCHITECTURE ---
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, num_classes=9, num_layers=2, dropout=0.3):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            bidirectional=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        pooled, _ = torch.max(lstm_out, dim=1)
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.dropout(out)
        logits = self.fc2(out)
        return logits

# --- 3. LOAD SAVED ARTIFACTS ---
with open('vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

# Instantiate model
VOCAB_SIZE = len(vocab.word2idx)
NUM_CLASSES = len(label_encoder.classes_)

model = BiLSTMClassifier(vocab_size=VOCAB_SIZE, num_classes=NUM_CLASSES).to(device)

# Load saved weights and set to evaluation mode
model.load_state_dict(torch.load('bilstm_toxicity_model.pth', map_location=device))
model.eval()

print("Model and vocabulary loaded successfully!")

Model and vocabulary loaded successfully!


In [4]:
def predict_toxicity(query, image_description, max_len=36):
    # 1. Format combined text using the 'sep' keyword
    full_text = f"{query} sep {image_description}"
    
    # 2. Encode to numerical tokens
    encoded = vocab.encode(full_text, max_len=max_len)
    input_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    
    # 3. Model Inference
    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = F.softmax(logits, dim=1)
        
        predicted_idx = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_idx].item()
        
    # 4. Map back to class name
    predicted_category = label_encoder.inverse_transform([predicted_idx])[0]
    
    print(f"Input Query : '{query}'")
    print(f"Image Desc  : '{image_description}'")
    print(f"Predicted   : {predicted_category} (Confidence: {confidence * 100:.2f}%)")
    print("-" * 60)
    
    return predicted_category

In [7]:
# Test 1: Harmless Query + Harmless Image -> Expected: Safe
predict_toxicity(
    query="How can I prepare for a software engineering job interview?", 
    image_description="A person sitting at a desk typing on a laptop."
)

# Test 2: Harmless Query + Suspicious Image -> Expected: Violent Crimes or Non-Violent Crimes
predict_toxicity(
    query="Where is the nearest convenience store open late?", 
    image_description="Police tape across a crime scene at night."
)

# Test 3: Unseen Cyber/Hate Query + Neutral Image -> Expected: Unknown S-Type or Non-Violent Crimes
predict_toxicity(
    query="Can you explain how SQL injection vulnerabilities are exploited by hackers?", 
    image_description="People walking through a peaceful city street."
)

# Test 4: Election Query + Neutral Image -> Expected: Elections
predict_toxicity(
    query="What is the exact treatment for my chronic back pain? ", 
    image_description="People running through a peaceful city garden."
)

Input Query : 'How can I prepare for a software engineering job interview?'
Image Desc  : 'A person sitting at a desk typing on a laptop.'
Predicted   : Non-Violent Crimes (Confidence: 60.14%)
------------------------------------------------------------
Input Query : 'Where is the nearest convenience store open late?'
Image Desc  : 'Police tape across a crime scene at night.'
Predicted   : Violent Crimes (Confidence: 99.69%)
------------------------------------------------------------
Input Query : 'Can you explain how SQL injection vulnerabilities are exploited by hackers?'
Image Desc  : 'People walking through a peaceful city street.'
Predicted   : Safe (Confidence: 86.68%)
------------------------------------------------------------
Input Query : 'What is the exact treatment for my chronic back pain? '
Image Desc  : 'People running through a peaceful city garden.'
Predicted   : Safe (Confidence: 91.91%)
------------------------------------------------------------


'Safe'